# Fundamentals 12 - Single AgenticSystem Completo

Un solo AgenticSystem con tools, agente, runtime, output final, lineage, environment y eval. El LM opcional explica, pero el camino determinista siempre corre.


In [ ]:
import os
import agentic_systems as lab
PRETTY = False
scheduler = lab.scheduler(timeout_s=60, max_retries=0, max_tool_calls=6, max_turns=6)
local_runtime = lab.runtime(provider="python-direct", model="local-python", region="local", scheduler=scheduler)
lm_runtime = lab.runtime(provider="auto", scheduler=scheduler)
lm_resolution = lm_runtime.describe()
force_local_only = bool(os.getenv("AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS"))
lm_available = lm_resolution["selected_provider"] != "auto" and not force_local_only
workspace = lab.AgenticSystem(model=lm_runtime.model_id or "local-python", region=lm_runtime.region_name or "local", runtime=lm_runtime)
USER_PROMPT = "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
NUMBERS = [10, 20, 9, 4, 2]
EXPECTED = 42
lab.show({"runtime_auto": lm_resolution, "lm_available": lm_available, "force_local_only": force_local_only})


In [ ]:
@lab.tool
def solve_arithmetic(numbers: list[int]) -> dict:
    a,b,c,d,e = numbers
    x1 = a + b; x2 = x1 - c; x3 = x2 * d; x4 = x3 / e
    return {"procedure": [f"{a}+{b}={x1}", f"{x1}-{c}={x2}", f"{x2}*{d}={x3}", f"{x3}/{e}={x4:g}"], "result": int(x4) if float(x4).is_integer() else x4}
@lab.tool
def judge_result(result: float, expected: float) -> dict:
    ok = float(result) == float(expected)
    return {"ok": ok, "score": 1.0 if ok else 0.0, "result": result, "expected": expected}


## 1) Sistema, contrato, policy y agente


In [ ]:
@lab.tool
def record_review(summary: str) -> dict:
    """Registra una revisi?n LM como evidencia estructurada."""
    return {"summary": summary}

single_system = lab.AgenticSystem(model="local-python", region="local", runtime=local_runtime)
policy = lab.RunPolicy(max_tool_calls=1, temperature=0.0, trace="compact")
contract = lab.AgentContract(must_call=["solve_arithmetic"], tool_expectation=lab.expect.exactly("solve_arithmetic"), completion="when_required_tools_satisfied")
agent = single_system.agent(name="single_solver", instructions="Resuelve el problema estructurado.", tools=[solve_arithmetic], engine="python-direct", runtime=local_runtime, contract=contract, policy=policy)
explainer = workspace.agent(name="single_lm_explainer", instructions="Explica la soluci?n sin cambiar n?meros.", tools=[record_review], runtime=lm_runtime, policy=lab.RunPolicy.for_mode("eval"))
lab.show({"system": single_system.inspect(), "agent": agent.info(), "explainer": explainer.info()})


## 2) Run completo + final_answer + lineage


In [ ]:
solve = agent.run({"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}})
explanation = None
explanation_result = None
if lm_available:
    explanation_result = explainer.run(str(solve.data))
    explanation = explanation_result.text
else:
    lab.show({"status": "skipped", "reason": lm_resolution["reason"]}, title="LM explainer saltado")

final = lab.final_answer(
    {"procedimiento": solve.data["procedure"], "resultado_final": solve.data["result"], "explicacion_lm": explanation},
    schema=lab.output_schema(fields=["procedimiento", "resultado_final", "explicacion_lm"]),
)
result = lab.compose_result(
    text="Single AgenticSystem completo ejecutado.",
    data=final,
    results=[solve, explanation_result],
    mode="single-agentic-system",
    input=USER_PROMPT,
    meta={"runtime_auto_resolution": lm_resolution},
)
lineage = result.lineage(name="fundamentals.single_agentic_system", question=USER_PROMPT, goal="Explicar el ciclo completo de un solo sistema.")
lab.human_result(result, title="Human result - Single AgenticSystem", pretty=PRETTY, show_lineage=True, lineage=lineage)

## 3) Environment y eval del mismo agente


In [ ]:
def transition_fn(row: dict, action: dict | None, info: dict) -> dict:
    out = agent.run({"tool": "solve_arithmetic", "input": {"numbers": row["numbers"]}}).data
    return {"result": out["result"], "expected": row["expected"], "ok": out["result"] == row["expected"]}

def reward_fn(state: dict) -> float:
    return 1.0 if state.get("ok") else 0.0

env_records = [{"numbers": NUMBERS, "expected": EXPECTED}]
env = lab.AgenticEnvironment(name="single_system_env", records=env_records, initial_memory={}, transition_fn=transition_fn, reward_fn=reward_fn)
env.reset()
_, reward, terminated, truncated, info = env.step()
report = lab.run_eval(agent, [{"id": "default", "input": {"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}}, "expected": {"result": EXPECTED}}])
lab.show({"env_summary": env.summary(), "step": info["transition"], "eval_report": report.to_dict()})


In [ ]:
lab.show({"notebook": "12_single agentic_system.ipynb", "api_coverage": ["AgenticSystem", "tool", "agent", "runtime(provider='auto')", "final_answer", "compose_result", "RunResult.lineage", "AgenticEnvironment", "run_eval"]})
